# 04 · Train

Notebook 03 built and explained three fusion architectures (early / intermediate / late) for
two tasks (classification, survival), sharing one MIL pooling module (`ABMIL`) and one head
architecture (`PredictionHead`) so the comparison isolates the fusion mechanism itself. This
notebook trains all six combinations on the real CHIMERA features from notebooks 01-02, using
the cross-validation folds already computed in notebook 01.

**Multi-slide patients, trained the [nnMIL](https://arxiv.org/abs/2511.14907) way:** CHIMERA
patients have 1-12 WSI slides each. Rather than merging slides *inside* the model, each
patient's label (`BCR` / `time_to_follow-up/BCR`) is copied onto *every one* of their slides,
and each slide becomes its own independent training row (with that patient's MRI + clinical
vectors repeated across their slides). Training and validation loss below are therefore
slide-level too -- no patient-level merging happens anywhere in this notebook.

**What this notebook does:** build a slide-level `Dataset`/`DataLoader`, train all 6
(fusion x task) combinations across the CV folds from notebook 01, log every run to
[Weights & Biases](https://wandb.ai), and checkpoint the best-val-loss model per
(fusion, task, fold) under `data/checkpoints/`.

**What this notebook doesn't do:** aggregate slide-level predictions back to patient level
(mean probability + majority vote for classification, mean risk score for survival -- matching
nnMIL's actual evaluation convention) or compute AUROC / C-index. That's a separate
inference/evaluation notebook, built later, using the checkpoints saved here.

**Prerequisites:** notebooks 00-02 fully run (WSI + MRI features under `data/features/`,
clinical embeddings under `data/prepared/clinical_embeddings/`, label CSVs under
`data/labels/`), and a [Weights & Biases](https://wandb.ai) account with an API key in `.env`
(see `.env.example`).

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> repo root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os

import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader, Dataset

from src.fusion_models import (
    EarlyFusionModel,
    IntermediateFusionModel,
    LateFusionModel,
    NLLLoss,
    discretize_time,
)

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
# --- Project paths -----------------------------------------------------------
LABELS_DIR = PROJECT_ROOT / "data" / "labels"
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

CLINICAL_EMBEDDING_DIR = PREPARED_DIR / "clinical_embeddings"
WSI_FEATURE_DIR = FEATURES_DIR / "wsi"
MRI_FEATURE_DIR = FEATURES_DIR / "mri"

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")
wandb_key = os.environ.get("WANDB_API_KEY")
if not wandb_key:
    raise RuntimeError(
        "WANDB_API_KEY not set. Copy .env.example to .env and add a key from "
        "https://wandb.ai/authorize."
    )
wandb.login(key=wandb_key)
WANDB_PROJECT = "minicourse-multimodal-bcr"

## Step 1 · Load train labels + CV folds

`train_labels.csv` (from notebook 01) has one row per *patient*, with a pre-computed
stratified `fold` column -- we reuse it directly for cross-validation instead of re-splitting
here, so every fusion strategy x task combination trains and validates on the exact same
patient splits.

In [ ]:
train_labels = pd.read_csv(LABELS_DIR / "train_labels.csv")
folds = sorted(train_labels["fold"].unique())
print(f"{len(train_labels)} training patients across {len(folds)} folds")
train_labels.head()

## Step 2 · Slide-level dataset

Each patient contributes one `(N patches, 1536)` WSI feature file *per slide*
(`data/features/wsi/{patient_id}_{slide_id}_features.npy`, from notebook 02).
`SlideLevelDataset` expands a list of patient IDs into one row *per slide* -- repeating that
patient's MRI embedding, clinical embedding, and label onto every one of their slides, so a
patient with 3 slides contributes 3 independent training rows, each with the same label.

In [ ]:
class SlideLevelDataset(Dataset):
    """One row per (patient, slide). A patient's MRI/clinical vectors and label are repeated
    across every one of their slides -- see the multi-slide note in this notebook's intro."""
    def __init__(self, patient_ids: list[str], labels_df: pd.DataFrame):
        self.labels_df = labels_df.set_index("patient_id")
        self.rows = [
            (patient_id, wsi_path)
            for patient_id in patient_ids
            for wsi_path in sorted(WSI_FEATURE_DIR.glob(f"{patient_id}_*_features.npy"))
        ]

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        patient_id, wsi_path = self.rows[idx]
        label = self.labels_df.loc[patient_id]
        mri_path = next(MRI_FEATURE_DIR.glob(f"{patient_id}_*_features.npy"))
        clinical_path = CLINICAL_EMBEDDING_DIR / f"{patient_id}_embedding.npy"

        return {
            "patient_id": patient_id,
            "wsi": torch.from_numpy(np.load(wsi_path)).float(),
            "mri": torch.from_numpy(np.load(mri_path)).float(),
            "clinical": torch.from_numpy(np.load(clinical_path)).float(),
            "y_time": torch.tensor(label["time_to_follow-up/BCR"], dtype=torch.float32),
            "y_event": torch.tensor(label["BCR"], dtype=torch.float32),
        }


def collate_slides(samples: list[dict]) -> dict:
    """Matches the batch dict shape EarlyFusionModel/IntermediateFusionModel/LateFusionModel
    expect: batch["wsi"] stays a list (patch count varies per slide); MRI, clinical, and
    labels stack into fixed-size tensors."""
    return {
        "patient_id": [s["patient_id"] for s in samples],
        "wsi": [s["wsi"] for s in samples],
        "mri": torch.stack([s["mri"] for s in samples]),
        "clinical": torch.stack([s["clinical"] for s in samples]),
        "y_time": torch.stack([s["y_time"] for s in samples]),
        "y_event": torch.stack([s["y_event"] for s in samples]),
    }

In [ ]:
# Sanity check: how many slide-level rows does the full training set expand to?
_all_rows = SlideLevelDataset(train_labels["patient_id"].tolist(), train_labels)
print(f"{len(train_labels)} patients -> {len(_all_rows)} slide-level training rows")

_sample = _all_rows[0]
{k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in _sample.items()}

## Step 3 · Train all (fusion x task) combinations across folds

For each of the 3 fusion strategies x 2 tasks x `len(folds)` CV folds, train independently:
split patients into train/val by the `fold` column, wrap them in slide-level `DataLoader`s,
train for `NUM_EPOCHS`, log train/val loss to Weights & Biases every epoch, and checkpoint the
best-val-loss model. Everything here stays slide-level -- no patient aggregation.

In [ ]:
FUSION_CLASSES = {
    "early": EarlyFusionModel,
    "intermediate": IntermediateFusionModel,
    "late": LateFusionModel,
}
TASKS = ["classification", "survival"]

NUM_TIME_BINS = 15
MAX_TIME = float(train_labels["time_to_follow-up/BCR"].max())
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4

In [ ]:
def move_batch_to_device(batch: dict, device: str) -> dict:
    """batch["wsi"] is a list of variable-shape tensors, so it needs its own loop --
    everything else is already a fixed-size tensor Tensor.to() handles directly."""
    return {
        "wsi": [w.to(device) for w in batch["wsi"]],
        "mri": batch["mri"].to(device),
        "clinical": batch["clinical"].to(device),
        "y_time": batch["y_time"].to(device),
        "y_event": batch["y_event"].to(device),
    }


def compute_loss(model_out: torch.Tensor, batch: dict, task: str) -> torch.Tensor:
    if task == "classification":
        return nn.functional.binary_cross_entropy_with_logits(model_out.squeeze(1), batch["y_event"])
    y_time_bins = discretize_time(batch["y_time"], NUM_TIME_BINS, MAX_TIME, model_out.device)
    return NLLLoss()(model_out, y_time_bins, batch["y_event"])


def run_epoch(model: nn.Module, loader: DataLoader, task: str, optimizer=None) -> float:
    """One pass over `loader`. Trains (and backprops) when `optimizer` is given, otherwise
    just evaluates -- used for the validation loader."""
    model.train() if optimizer is not None else model.eval()
    losses = []
    for batch in loader:
        batch = move_batch_to_device(batch, DEVICE)
        with torch.set_grad_enabled(optimizer is not None):
            out = model(batch)
            loss = compute_loss(out, batch, task)
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        losses.append(loss.item())
    return sum(losses) / len(losses)

In [ ]:
for fusion_name, ModelClass in FUSION_CLASSES.items():
    for task in TASKS:
        for fold in folds:
            val_mask = train_labels["fold"] == fold
            train_patients = train_labels.loc[~val_mask, "patient_id"].tolist()
            val_patients = train_labels.loc[val_mask, "patient_id"].tolist()

            train_loader = DataLoader(
                SlideLevelDataset(train_patients, train_labels),
                batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collate_slides,
            )
            val_loader = DataLoader(
                SlideLevelDataset(val_patients, train_labels),
                batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_slides,
            )

            model = ModelClass(task=task, num_time_bins=NUM_TIME_BINS).to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

            wandb.init(
                project=WANDB_PROJECT,
                group=f"{fusion_name}-{task}",
                name=f"{fusion_name}-{task}-fold{fold}",
                config={
                    "fusion": fusion_name, "task": task, "fold": fold,
                    "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
                },
                reinit=True,
            )

            best_val_loss = float("inf")
            ckpt_path = CHECKPOINT_DIR / f"{fusion_name}_{task}_fold{fold}.pt"

            for epoch in range(NUM_EPOCHS):
                train_loss = run_epoch(model, train_loader, task, optimizer)
                val_loss = run_epoch(model, val_loader, task, optimizer=None)
                wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save({
                        "model_state_dict": model.state_dict(),
                        "epoch": epoch,
                        "val_loss": val_loss,
                        "fusion": fusion_name,
                        "task": task,
                        "fold": fold,
                        "num_time_bins": NUM_TIME_BINS,
                        "max_time": MAX_TIME,
                    }, ckpt_path)

            wandb.finish()
            print(f"{fusion_name}/{task}/fold{fold}: best val_loss={best_val_loss:.4f} -> {ckpt_path.name}")

## Summary

This notebook trained all 3 fusion strategies x 2 tasks x `len(folds)` CV folds as
slide-level models -- each patient's slides are independent training rows sharing that
patient's label -- logging every run to Weights & Biases and saving the best-val-loss
checkpoint per (fusion, task, fold) under `data/checkpoints/`.

**Next up:** a separate inference/evaluation notebook loads these checkpoints, runs each model
over its validation (and held-out test) patients, aggregates slide-level predictions back to
patient level -- mean probability + majority vote for classification, mean risk score for
survival, matching nnMIL's own convention -- and reports AUROC / C-index per fusion strategy.